# Conditional CGB paths: from observations to a frozen forecast

Read [the full derivation](../docs/10-structural-model-derivation.md) alongside this notebook. Source inspiration supplies questions, not predetermined state counts, matrices or probability laws. The account asks about outright CGB movement over one to four hours.

Direction, expected movement and adverse excursion come from a weighted distribution over complete future midpoint paths. No market data is included. Run All without inputs reports awaiting_data. The notebook imports the canonical package so command-line and notebook behavior agree. The older CAD notebook remains a separate baseline.


In [ ]:
from pathlib import Path
import sys
PROJECT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                if (p / 'pyproject.toml').exists() and (p / 'src' / 'momentum').is_dir()), None)
if PROJECT is None:
    raise RuntimeError('Open from the momentum project or its notebooks folder.')
sys.path.insert(0, str(PROJECT / 'src'))
import pandas as pd
from momentum import (ResearchConfig, ScenarioConfig, prepare_panel, build_features,
    build_state_observations, fit_scenario_research, predict_scenarios,
    save_deployment, load_deployment, save_live_reading, read_live_reading, matured_feedback)


## 1. Declare the environment and information cutoff

Supply actual pandas tables following [the input contract](../docs/11-running-and-testing.md). AS_OF is the explicit receiver cutoff, even when a feed is stale. SESSIONS supplies the permitted session and close. The intended experiment selects CGB, US duration and the Canadian curve. A smaller CGB-only study must be declared explicitly. Optional swaps, OIS, forwards, book, VWAP and context need actual history. Signed flow requires upstream aggressor classification.


In [ ]:
QUOTES = None
RATES = None
SESSIONS = None
TRADES = None
CONTEXT = None
AS_OF = None
CONFIG = ResearchConfig(feature_modules=('cgb', 'us', 'curve'))
SCENARIO = ScenarioConfig()
INPUTS_READY = QUOTES is not None and SESSIONS is not None and AS_OF is not None


## 2. Construct what was actually observable

$$
\mathcal F_t=\sigma\{x_i:a_i\le t\}.
$$

Receiver-time quote selection, past-only scale and contract/session segmentation establish the measurement boundary. Raw CGB movement, common duration and Canadian deviation stay visible separately.


In [ ]:
PANEL = FEATURES = None
OBSERVATIONS = pd.DataFrame()
if INPUTS_READY:
    PANEL = prepare_panel(QUOTES, RATES, SESSIONS, CONFIG,
        trades=TRADES, context=CONTEXT, as_of=pd.Timestamp(AS_OF))
    FEATURES = build_features(PANEL, CONFIG)
    OBSERVATIONS = build_state_observations(PANEL, FEATURES, CONFIG, SCENARIO, TRADES)
    display(OBSERVATIONS.tail(12))
else:
    print('awaiting_data: QUOTES, SESSIONS and timezone-aware AS_OF are required.')


## 3. Inspect state and coverage before fitting

Five states describe completed motion: balanced, up-responsive, up-weakening, down-responsive, down-weakening. State age preserves persistence. These describe history, not causes or known future outcomes.

Missing classified flow leaves absorption unavailable. A recovery failure is timestamped only when its subsequent retreat has actually occurred. Reference availability does not itself establish a stable economic relation.


In [ ]:
if not OBSERVATIONS.empty:
    display(OBSERVATIONS.groupby('session_id').agg(
        decisions=('valid', 'size'), valid=('valid', 'sum'),
        classified_flow=('flow_available', 'sum'),
        reference=('reference_available', 'sum'), vwap=('vwap_available', 'sum')))
    display(pd.crosstab(OBSERVATIONS.session_id, OBSERVATIONS.state_id))
else:
    print('No states or coverage statistics yet.')


## 4. Fit only on completed training paths

$$
Z_{i,u}=\frac{m_{t_i+u}-m_{t_i}}{\tau\sigma_{t_i}},
\qquad Y_{t,u}^{(i)}=\sigma_t Z_{i,u}.
$$

TRAIN fits robust similarity coordinates, horizon-specific state transitions, a future-state tree head and a future-price tree head. CAL fits only the state pool and path-expert weights. TEST reports untouched subsequent outcomes.

Each horizon uses its own complete path bank. A label cannot cross its partition, session, invalid quote gap or contract boundary. Local scale transfer is an explicit hypothesis. A finite bank cannot manufacture an unseen tail shape.


In [ ]:
RESEARCH = fit_scenario_research(PANEL, FEATURES, CONFIG, SCENARIO, TRADES)
print('Research status:', RESEARCH['status'])
if 'reason' in RESEARCH:
    print(RESEARCH['reason'])
for horizon, item in RESEARCH['horizons'].items():
    print(horizon, item['status'], item.get('reason', ''), item.get('eligible_sessions', {}))


## 5. Combine distributions on one scenario space

$$
\pi_i(t)=\sum_{e=1}^{3}\omega_{e,h}w_i^{(e)}(t),
\qquad \omega_{e,h}\ge0,\quad\sum_e\omega_{e,h}=1.
$$

The local expert uses similarity. The state expert redistributes that mass by forecast future state; the direct expert uses forecast endpoint class. A convex mixture avoids an unsupported independence assumption. All path summaries use the same final mass. Exact sums require no Monte Carlo simulation.

Inspect held-out probability scores, endpoint error, per-session losses, reliability and risk-quantile coverage. Individual experts and a training-frequency prior provide controls. Repeated chronological periods, persistence-only controls and costed trading tests remain empirical work.


In [ ]:
if RESEARCH['status'] in ('ready', 'partial'):
    display(RESEARCH['report'])
    for horizon, item in RESEARCH['horizons'].items():
        if item['status'] != 'ready':
            continue
        print('Horizon:', horizon, 'State-ML weight:', item['state_ml_weight'])
        print('Local / structural / supervised weights:', item['mixture_weights'])
        print('Risk quantile diagnostics:', item['risk_metrics'])
        display(item['metrics']['mixture']['per_session'])
        display(item['metrics']['mixture']['reliability'])
        display(item['test_predictions'].tail())
else:
    print('No fitted market results exist.')


## 6. Produce a research reading

The reading carries its actual decision time. A horizon extending past the session close is unavailable. Entropy, expert disagreement, missing measurements and scenario concentration describe different uncertainties. Effective scenarios are not independent days.

These are midpoint forecasts. Price-side costs belong to a declared trade interpretation using the execution-accounting module. This notebook selects no position, stop, passive fill or broker order.


In [ ]:
READINGS = pd.DataFrame()
if RESEARCH['status'] in ('ready', 'partial'):
    READINGS = predict_scenarios(RESEARCH, PANEL, FEATURES, TRADES)
    display(READINGS)
else:
    print('No live reading: awaiting data or eligible history.')


## 7. Freeze a named experiment before consumption

Set DEPLOYMENT_DIR to a new empty directory only when saving an identified fit. Keep it blank while studying. Saved artifacts contain TRAIN banks, configuration, models, hashes and held-out diagnostics. Loading verifies package and artifact hashes. Dependency versions are recorded.

A published reading cannot overwrite a previous file. The final consumer only reads persisted JSON; it cannot refit or reconstruct a state. The runs folder is ignored by Git because fitted artifacts can contain private market data.


In [ ]:
DEPLOYMENT_DIR = None  # Example: PROJECT / 'runs' / 'research-001'
READING_FILE = None    # Example: PROJECT / 'runs' / 'readings' / 'decision-001.json'
DEPLOYMENT = None
if DEPLOYMENT_DIR is not None and RESEARCH['status'] in ('ready', 'partial'):
    MANIFEST = save_deployment(RESEARCH, DEPLOYMENT_DIR)
    DEPLOYMENT = load_deployment(DEPLOYMENT_DIR)
    FROZEN_READINGS = predict_scenarios(DEPLOYMENT, PANEL, FEATURES, TRADES)
    if READING_FILE is not None:
        save_live_reading(FROZEN_READINGS, DEPLOYMENT, READING_FILE)
    print('Saved and verified:', MANIFEST['run_id'])
else:
    print('No deployment written. Supply real data and a new run directory.')


In [ ]:
if READING_FILE is not None and Path(READING_FILE).is_file():
    display(read_live_reading(READING_FILE))
else:
    print('No persisted reading selected.')


## 8. Close the information loop after outcomes mature

Preserve original readings. Later, use matured_feedback with those predictions and a newly constructed receiver-time panel. It evaluates only complete horizons and never changes parameters. Store a separate outcome ledger. Changing the learning rule requires a new experiment.

Before interpreting a result:

- Does the target match outright CGB and the account's holding window?
- Are units, contracts, clocks and permitted sessions correct?
- Are missing flow and reference inputs visible?
- Are mechanism claims still hypotheses?
- Does every fitted object precede its evaluated outcome?
- Do direction and path risk use the same scenario mass?
- Is historical support adequate for the current event?
- Has the model beaten simple controls on new sessions?
- Has a specified trading rule survived actual costs?

The software checks can be run now. Market questions require real observations. See [the verification record](../audits/implementation-verification.md).
